# Gold Layer — Payment Summary

**Table**: `aiubereats.payments.gold_payment_summary`  
**Source**: `aiubereats.payments.silver_payment_events`  
**Purpose**: One row per `payment_id` summarising the full payment lifecycle.  
**Quality level**: Business-ready, tested, curated.

## Output Schema

| Column | Type | Description |
|--------|------|-------------|
| `payment_id` | STRING | Business key |
| `created_at` | TIMESTAMP | Timestamp of the `created` event |
| `authorized_at` | TIMESTAMP | Timestamp of the `authorized` event |
| `captured_at` | TIMESTAMP | Timestamp of the `captured` event |
| `payment_status` | STRING | `captured` / `authorized` / `created` |
| `auth_time_seconds` | DOUBLE | `authorized_at - created_at` in seconds (null if authorized_at is null) |
| `capture_time_seconds` | DOUBLE | `captured_at - authorized_at` in seconds (null if either is null) |
| `total_processing_time_seconds` | DOUBLE | `captured_at - created_at` in seconds (null if captured_at is null) |
| `event_count` | LONG | Total distinct events observed for this payment |
| `_computed_at` | TIMESTAMP | When this Gold row was last computed |

## payment_status Derivation

```
captured_at IS NOT NULL                          → "captured"    (all 3 events present or at minimum created + captured)
authorized_at IS NOT NULL AND captured_at IS NULL → "authorized"  (created + authorized, not yet captured)
created_at IS NOT NULL  (fallthrough)            → "created"     (only created event seen)
```

## Design Decisions

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Aggregation approach | `PIVOT` via conditional aggregation (`max(CASE WHEN …)`) | Spark does not support a dynamic PIVOT; conditional aggregation is equivalent, fully explicit, and Spark Connect compatible. |
| Write mode | `MERGE` on `payment_id` | Gold is refreshed on every Silver run. MERGE avoids full-table rewrites and handles incremental updates correctly. |
| Partitioning | None at Gold | `gold_payment_summary` has one row per `payment_id`. Partitioning a wide table with high cardinality keys produces too many small files. Liquid CLUSTER BY (payment_status) is used instead for efficient status-filtered queries. |
| `auth_time_seconds`, `capture_time_seconds`, `total_processing_time_seconds` | `DOUBLE` (seconds) | Three granular time metrics covering each phase of the payment lifecycle. DOUBLE preserves sub-second precision from `unix_timestamp()` arithmetic. |
| NULL handling in status | Explicit CASE/WHEN order | Captures the most progressed state. A payment that skips `authorized` and goes straight to `captured` (platform edge case) is still correctly classified as `captured`. |

## Serverless / Spark Connect Notes

- `unix_timestamp(col)` is a standard Spark SQL Column function, fully Spark Connect compatible.
- No UDFs, no RDDs, no `sparkContext` references.
- `DeltaTable.forName()` + `.merge()` is supported in Serverless Databricks (Delta is bundled).


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

In [ ]:
# ── Configuration — Databricks Widgets ───────────────────────────────────────────────
#
# Widget declarations are idempotent in Databricks — safe to re-run.

dbutils.widgets.text("silver_table", "aiubereats.payments.silver_payment_events")
dbutils.widgets.text("gold_table",   "aiubereats.payments.gold_payment_summary")

SILVER_TABLE = dbutils.widgets.get("silver_table")
GOLD_TABLE   = dbutils.widgets.get("gold_table")

In [ ]:
# ── Create Gold table (DDL) ───────────────────────────────────────────────────────
#
# No PARTITIONED BY: one row per payment_id; partitioning would create
# many tiny files. Liquid Clustering is used instead.
#
# CLUSTER BY (payment_status, payment_id):
#   - payment_status: dashboard queries filter by status; clustering enables
#     skip-reading irrelevant files without a full scan.
#   - payment_id: point-lookups and joins from downstream consumers.
#
# Three time metrics (all nullable — only populated when both endpoints exist):
#   auth_time_seconds             = authorized_at - created_at
#   capture_time_seconds          = captured_at   - authorized_at
#   total_processing_time_seconds = captured_at   - created_at
#
# TBLPROPERTIES:
#   autoOptimize.optimizeWrite  — coalesces small files on write
#   autoOptimize.autoCompact    — background compaction
#   enableDeletionVectors       — efficient soft-deletes when payments are corrected
#   logRetentionDuration        — 30 days is sufficient for a computed/refreshed table

spark.sql("""
    CREATE TABLE IF NOT EXISTS aiubereats.payments.gold_payment_summary (
        payment_id                     STRING     NOT NULL,
        created_at                     TIMESTAMP,
        authorized_at                  TIMESTAMP,
        captured_at                    TIMESTAMP,
        payment_status                 STRING     NOT NULL,
        auth_time_seconds              DOUBLE,
        capture_time_seconds           DOUBLE,
        total_processing_time_seconds  DOUBLE,
        event_count                    LONG       NOT NULL,
        _computed_at                   TIMESTAMP  NOT NULL
    )
    USING DELTA
    CLUSTER BY (payment_status, payment_id)
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite'       = 'true',
        'delta.autoOptimize.autoCompact'         = 'true',
        'delta.enableDeletionVectors'            = 'true',
        'delta.logRetentionDuration'             = 'interval 30 days',
        'delta.deletedFileRetentionDuration'     = 'interval 7 days'
    )
    COMMENT 'Gold layer: one row per payment_id summarising the full payment lifecycle.'
""")

In [ ]:
# ── Read from Silver ──────────────────────────────────────────────────────────

silver_df = spark.table(SILVER_TABLE)
print(f"Silver records: {silver_df.count():,}")

In [ ]:
# ── Build Gold aggregation ────────────────────────────────────────────────────────────
#
# Technique: conditional aggregation to pivot event_name into timestamp columns.
#
#   max(CASE WHEN event_name = 'created' THEN event_timestamp END)
#
# This picks the timestamp for that event type per payment_id.
# If a payment has multiple 'created' events (malformed source), max() picks
# the latest — a conservative choice that matches the most-recent state.
#
# payment_status logic (CASE/WHEN priority order):
#   1. If captured_at IS NOT NULL  → "captured"    (terminal success state)
#   2. If authorized_at IS NOT NULL → "authorized"  (in-flight, past auth)
#   3. Otherwise                   → "created"     (only initiated)
#
# Three time metrics (all DOUBLE, seconds):
#   auth_time_seconds             = authorized_at - created_at
#                                   NULL if authorized_at is null
#   capture_time_seconds          = captured_at - authorized_at
#                                   NULL if either is null
#   total_processing_time_seconds = captured_at - created_at
#                                   NULL if captured_at is null
#
# unix_timestamp() returns epoch seconds as DOUBLE; subtraction yields elapsed
# seconds as DOUBLE, preserving sub-second precision.

gold_df = (
    silver_df
    .groupBy("payment_id")
    .agg(
        F.max(
            F.when(F.col("event_name") == "created", F.col("event_timestamp"))
        ).alias("created_at"),
        F.max(
            F.when(F.col("event_name") == "authorized", F.col("event_timestamp"))
        ).alias("authorized_at"),
        F.max(
            F.when(F.col("event_name") == "captured", F.col("event_timestamp"))
        ).alias("captured_at"),
        F.count("event_id").alias("event_count"),
    )
    # Derive payment_status (priority: captured > authorized > created)
    .withColumn(
        "payment_status",
        F.when(F.col("captured_at").isNotNull(),    F.lit("captured"))
         .when(F.col("authorized_at").isNotNull(),  F.lit("authorized"))
         .otherwise(                                F.lit("created"))
    )
    # ── Three time metrics ────────────────────────────────────────────────────────────
    # auth_time_seconds: time from creation to authorization
    .withColumn(
        "auth_time_seconds",
        F.when(
            F.col("authorized_at").isNotNull() & F.col("created_at").isNotNull(),
            F.unix_timestamp(F.col("authorized_at")) - F.unix_timestamp(F.col("created_at"))
        ).otherwise(F.lit(None).cast("double"))
    )
    # capture_time_seconds: time from authorization to capture
    .withColumn(
        "capture_time_seconds",
        F.when(
            F.col("captured_at").isNotNull() & F.col("authorized_at").isNotNull(),
            F.unix_timestamp(F.col("captured_at")) - F.unix_timestamp(F.col("authorized_at"))
        ).otherwise(F.lit(None).cast("double"))
    )
    # total_processing_time_seconds: end-to-end time from creation to capture
    .withColumn(
        "total_processing_time_seconds",
        F.when(
            F.col("captured_at").isNotNull() & F.col("created_at").isNotNull(),
            F.unix_timestamp(F.col("captured_at")) - F.unix_timestamp(F.col("created_at"))
        ).otherwise(F.lit(None).cast("double"))
    )
    # Add computation timestamp
    .withColumn("_computed_at", F.current_timestamp())
    # Final column order matching DDL
    .select(
        "payment_id",
        "created_at",
        "authorized_at",
        "captured_at",
        "payment_status",
        "auth_time_seconds",
        "capture_time_seconds",
        "total_processing_time_seconds",
        "event_count",
        "_computed_at",
    )
)

In [ ]:
# ── Gold quality check before write ──────────────────────────────────────────────────
#
# Validate the derived data before persisting.
# These are assertions, not filters. A failure here means Silver has bad data
# that slipped through the Silver quality gate and must be investigated.

gold_count = gold_df.count()
print(f"Gold rows to write: {gold_count:,}")

# Check 1: No null payment_id should exist (was enforced in Silver)
null_payment_ids = gold_df.filter(F.col("payment_id").isNull()).count()
assert null_payment_ids == 0, f"Gold quality failure: {null_payment_ids} rows with null payment_id"

# Check 2: payment_status must only contain valid values
invalid_statuses = gold_df.filter(
    ~F.col("payment_status").isin(["created", "authorized", "captured"])
).count()
assert invalid_statuses == 0, f"Gold quality failure: {invalid_statuses} rows with invalid payment_status"

# Check 3: all three time metrics must be non-negative when not null
for time_col in ["auth_time_seconds", "capture_time_seconds", "total_processing_time_seconds"]:
    negative_time = gold_df.filter(
        F.col(time_col).isNotNull() & (F.col(time_col) < 0)
    ).count()
    if negative_time > 0:
        print(f"WARNING: {negative_time} payments have negative {time_col}. "
              "This may indicate out-of-order event delivery in the source.")

# Check 4: Every captured payment must have a created_at
captured_without_created = gold_df.filter(
    (F.col("payment_status") == "captured") & F.col("created_at").isNull()
).count()
if captured_without_created > 0:
    print(f"WARNING: {captured_without_created} captured payments are missing created_at. "
          "Possible missing Bronze files for these payment_ids.")

print("Gold quality checks complete.")

In [ ]:
# ── Write to Gold — MERGE on payment_id ───────────────────────────────────────
#
# MERGE is used rather than overwrite to:
#   1. Support incremental updates (only changed payment_ids need recomputation).
#   2. Preserve Gold row history in the Delta log for time travel.
#   3. Avoid a full table rewrite on every Silver batch.
#
# The MERGE updates all columns unconditionally when there's a match:
#   payment_status can change (e.g., created → authorized → captured over time),
#   so all columns must be refreshed each time.

if spark.catalog.tableExists(GOLD_TABLE):
    gold_delta = DeltaTable.forName(spark, GOLD_TABLE)

    (
        gold_delta.alias("t")
        .merge(
            gold_df.alias("s"),
            "t.payment_id = s.payment_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete into {GOLD_TABLE}")
else:
    # First run after DDL creation
    (
        gold_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TABLE)
    )
    print(f"Initial load complete into {GOLD_TABLE}")

In [ ]:
# ── Verify and display summary ────────────────────────────────────────────────
gold = spark.table(GOLD_TABLE)
print(f"Gold row count: {gold.count():,}")

print("\nPayment status distribution:")
gold.groupBy("payment_status").count().orderBy("payment_status").display()

print("\nSample Gold rows:")
gold.orderBy("payment_id").limit(10).display()

In [ ]:
# ── Average time metrics by payment cohort ────────────────────────────────────────
#
# Useful sanity check: captured payments should have realistic processing
# times (seconds to minutes for payment processing).

(
    gold
    .filter(F.col("total_processing_time_seconds").isNotNull())
    .agg(
        F.min("total_processing_time_seconds").alias("min_total_seconds"),
        F.avg("total_processing_time_seconds").alias("avg_total_seconds"),
        F.max("total_processing_time_seconds").alias("max_total_seconds"),
        F.avg("auth_time_seconds").alias("avg_auth_seconds"),
        F.avg("capture_time_seconds").alias("avg_capture_seconds"),
        F.count("payment_id").alias("captured_payments")
    )
    .display()
)

In [ ]:
# ── Post-Gold OPTIMIZE ────────────────────────────────────────────────────────
#
# Compact files and apply liquid clustering on payment_status.
# Dashboard queries typically filter by payment_status, so clustering
# on this column dramatically reduces data scanned.

spark.sql(f"OPTIMIZE {GOLD_TABLE}")
print("OPTIMIZE complete.")